# Mini-TP 7 · Seguridad con SAIF — REST (FastAPI)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Capa elegida: **REST (FastAPI)**, sobre **mi propio modelo** de predicción de stroke (el mismo de los Mini-TP 1 a 3, cargado desde `common/`).

### SAIF en esta capa
**Riesgo:** abuso, inyección, DoS y *scraping* para robar el modelo.
**Controles:** autenticación (API key/JWT) · validación de entrada · *rate limiting*.
**Elementos SAIF:** bases sólidas · automatizar defensas.
**Gobernanza agregada:** log de auditoría + model card.

> Corre de punta a punta sin servicios externos (usa `TestClient` de FastAPI).

### Dependencias
```bash
pip install fastapi httpx pyjwt scikit-learn pandas numpy joblib
```

In [1]:
import sys, time, json, hashlib, datetime
from pathlib import Path
from typing import Literal

import numpy as np

def encontrar_common():
    candidatos = [Path.cwd()/"common", Path.cwd().parent/"common", Path.cwd().parent.parent/"common"]
    for c in candidatos:
        if (c/"artifacts"/"model.pkl").exists():
            return c
    raise FileNotFoundError("No encontré common/artifacts (corré: python common/train_model.py)")

COMMON = encontrar_common()
sys.path.insert(0, str(COMMON.parent))
from common.inference import load_model

modelo = load_model()          # mismo RandomForest + preprocesamiento (one-hot + escalado) de los Mini-TP 1-3
print("Modelo listo:", modelo.name, "v", modelo.version, "| métricas:", modelo.metrics)

Modelo listo: stroke_prediction_model_prod v 1 | métricas: {'accuracy': 0.9660668380462725, 'precision': 0.9758403361344538, 'recall': 0.9557613168724279, 'f1_score': 0.9656964656964657, 'train_observations': 7777, 'test_observations': 1945}


## 1. Versión VULNERABLE — cualquiera consulta el modelo, sin límites

In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

VALID = {"gender": "Male", "age": 67, "hypertension": 0, "heart_disease": 1, "ever_married": "Yes",
         "work_type": "Private", "Residence_type": "Urban", "avg_glucose_level": 228.69,
         "bmi": 36.6, "smoking_status": "smokes"}

app_v = FastAPI()
@app_v.post("/predict")
def predict_v(payload: dict):
    stroke, label, proba = modelo.predict_one(payload)
    return {"pred": stroke, "probability": proba}

cli_v = TestClient(app_v)
# sin credenciales, sin validar, sin límite: todo pasa
print("sin auth       :", cli_v.post("/predict", json=VALID).status_code, "(sin control: pasa con 200)")
print("entrada absurda:", cli_v.post("/predict", json={**VALID, "age": -5, "bmi": 9999}).status_code, "(sin validación: pasa con 200)")
print("1000 consultas seguidas: permitidas -> riesgo de extracción del modelo (scraping)")

c:\Users\Admin\anaconda3\envs\mlops2\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


sin auth       : 200 (sin control: pasa con 200)
entrada absurda: 200 (sin validación: pasa con 200)
1000 consultas seguidas: permitidas -> riesgo de extracción del modelo (scraping)


## 2. Control principal (GUIADO) — API key + validación + rate limiting
Versión mínima del control, antes de pasar a JWT con roles.

In [3]:
from fastapi import Depends, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, Field

API_KEY = "demo-key-123"          # en producción: desde variable de entorno
app_k = FastAPI()

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
def auth_key(k: str = Depends(api_key_header)):
    if k != API_KEY:
        raise HTTPException(401, "no autorizado")
    return "cliente"

_hits_k = {}
def rate_limit_k(cliente: str = Depends(auth_key)):
    now = time.time(); w = _hits_k.setdefault(cliente, [])
    w[:] = [t for t in w if now - t < 60]
    if len(w) >= 5:
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return cliente

class EntradaK(BaseModel):
    gender: Literal["Male", "Female", "Other"]
    age: float = Field(ge=0, le=120)
    hypertension: int = Field(ge=0, le=1)
    heart_disease: int = Field(ge=0, le=1)
    ever_married: Literal["Yes", "No"]
    work_type: Literal["Private", "Self-employed", "Govt_job", "children", "Never_worked"]
    Residence_type: Literal["Urban", "Rural"]
    avg_glucose_level: float = Field(ge=0, le=350)
    bmi: float = Field(ge=0, le=100)
    smoking_status: Literal["formerly smoked", "never smoked", "smokes", "Unknown"]

@app_k.post("/predict")
def predict_k(x: EntradaK, cliente: str = Depends(rate_limit_k)):
    stroke, label, proba = modelo.predict_one(x.model_dump())
    return {"prediccion": stroke, "probabilidad": proba}

cli = TestClient(app_k); h = {"X-API-Key": API_KEY}
print("sin key   :", cli.post("/predict", json=VALID).status_code, "(401)")
print("con key   :", cli.post("/predict", json=VALID, headers=h).status_code, "(200)")
print("mala forma:", cli.post("/predict", json={**VALID, "age": -5}, headers=h).status_code, "(422)")
codes = [cli.post("/predict", json=VALID, headers=h).status_code for _ in range(7)]
print("rate limit:", codes, "(429 al superar 5/min)")

sin key   : 401 (401)
con key   : 200 (200)
mala forma: 422 (422)
rate limit: [200, 200, 200, 429, 429, 429, 429] (429 al superar 5/min)


## 3. Mi entrega — JWT con roles + auditoría + entrada adaptada a mi modelo
- **TODO 1:** `requiere_rol()` con `HTTPBearer` + `jwt.decode(..., algorithms=["HS256"])`; 401 si falta/expira/está manipulado el token, 403 si el rol no está en `{"analista","admin"}`.
- **TODO 2:** `AUDITORIA` registra ts, usuario, **hash** de la entrada (no el dato clínico) y predicción.
- **TODO 3:** `Entrada` usa las features reales de mi modelo (`Literal` y rangos con `Field(ge=, le=)`).
- **Gobernanza:** endpoint `/v1/model-card`.

In [6]:
import sys; print(sys.executable)

c:\Users\Admin\anaconda3\envs\mlops2\python.exe


In [7]:
import os, jwt   # pip install pyjwt
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer

JWT_SECRET = os.environ.get("JWT_SECRET", "cambiar-en-produccion-con-32-bytes-o-mas")
ROLES_PERMITIDOS = {"analista", "admin"}
RATE_LIMIT = int(os.environ.get("RATE_LIMIT", "5"))
RATE_WINDOW = 60
AUDITORIA = []

def crear_token(usuario, rol, minutos=15):
    payload = {"sub": usuario, "rol": rol,
               "exp": datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=minutos)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

bearer = HTTPBearer(auto_error=False)

# TODO 1
def requiere_rol(cred: HTTPAuthorizationCredentials = Depends(bearer)) -> str:
    if cred is None:
        raise HTTPException(401, "falta token Bearer")
    try:
        datos = jwt.decode(cred.credentials, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "token inválido o expirado")
    if datos.get("rol") not in ROLES_PERMITIDOS:
        raise HTTPException(403, "rol sin permiso")
    return datos["sub"]

_hits = {}
def rate_limit(usuario: str = Depends(requiere_rol)) -> str:
    now = time.time(); w = _hits.setdefault(usuario, [])
    w[:] = [t for t in w if now - t < RATE_WINDOW]
    if len(w) >= RATE_LIMIT:
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return usuario

# TODO 3
class Entrada(EntradaK):
    """Features reales del modelo de stroke, con tipos y rangos estrictos."""

# TODO 2
def auditar(usuario, entrada, prediccion, probabilidad):
    AUDITORIA.append({
        "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "usuario": usuario,
        "hash_entrada": hashlib.sha256(json.dumps(entrada, sort_keys=True).encode()).hexdigest(),
        "prediccion": prediccion,
        "probabilidad": round(probabilidad, 4),
        "modelo_version": modelo.version,
    })

app_s = FastAPI(title="Stroke Predictor API (segura)")

@app_s.post("/v1/predict")
def predict_s(x: Entrada, usuario: str = Depends(rate_limit)):
    entrada = x.model_dump()
    stroke, label, proba = modelo.predict_one(entrada)
    auditar(usuario, entrada, stroke, proba)
    return {"stroke_predicted": stroke, "label": label, "probability": proba,
            "model_name": modelo.name, "model_version": modelo.version}

@app_s.get("/v1/model-card")
def model_card(_: str = Depends(requiere_rol)):
    return {
        "nombre": modelo.name, "version": modelo.version,
        "tipo": modelo.data_dict["model_type"], "metricas": modelo.metrics,
        "uso_previsto": "Apoyo a la priorización de pacientes con riesgo de stroke; no es diagnóstico.",
        "limitaciones": "Dataset de Kaggle (~5k registros, clase positiva ~5%, balanceada con SMOTE).",
        "controles": ["JWT + roles", "validación Pydantic", "rate limiting", "auditoría por hash"],
    }

print("API segura definida")

API segura definida


### Prueba de los controles

In [8]:
cli_s = TestClient(app_s)
auth = lambda t: {"Authorization": f"Bearer {t}"}
analista = crear_token("cecilia", "analista")
intruso  = crear_token("mallory", "invitado")
expirado = crear_token("cecilia", "analista", minutos=-1)

print("sin token          :", cli_s.post("/v1/predict", json=VALID).status_code, "(401)")
print("token expirado     :", cli_s.post("/v1/predict", json=VALID, headers=auth(expirado)).status_code, "(401)")
print("token manipulado   :", cli_s.post("/v1/predict", json=VALID, headers=auth(analista + "x")).status_code, "(401)")
print("rol no permitido   :", cli_s.post("/v1/predict", json=VALID, headers=auth(intruso)).status_code, "(403)")
print("entrada inválida   :", cli_s.post("/v1/predict", json={**VALID, "age": -5}, headers=auth(analista)).status_code, "(422)")
r = cli_s.post("/v1/predict", json=VALID, headers=auth(analista))
print("analista, válido   :", r.status_code, r.json(), "(200)")
codes = [cli_s.post("/v1/predict", json=VALID, headers=auth(analista)).status_code for _ in range(6)]
print("rate limit (ráfaga):", codes, f"(429 al superar {RATE_LIMIT}/min en total)")
print("model card         :", cli_s.get("/v1/model-card", headers=auth(analista)).json())

sin token          : 401 (401)
token expirado     : 401 (401)
token manipulado   : 401 (401)
rol no permitido   : 403 (403)
entrada inválida   : 422 (422)
analista, válido   : 200 {'stroke_predicted': False, 'label': 'Not likely to have a stroke', 'probability': 0.15, 'model_name': 'stroke_prediction_model_prod', 'model_version': 1} (200)
rate limit (ráfaga): [200, 200, 200, 429, 429, 429] (429 al superar 5/min en total)
model card         : {'nombre': 'stroke_prediction_model_prod', 'version': 1, 'tipo': 'RandomForestClassifier', 'metricas': {'accuracy': 0.9660668380462725, 'precision': 0.9758403361344538, 'recall': 0.9557613168724279, 'f1_score': 0.9656964656964657, 'train_observations': 7777, 'test_observations': 1945}, 'uso_previsto': 'Apoyo a la priorización de pacientes con riesgo de stroke; no es diagnóstico.', 'limitaciones': 'Dataset de Kaggle (~5k registros, clase positiva ~5%, balanceada con SMOTE).', 'controles': ['JWT + roles', 'validación Pydantic', 'rate limiting', 'audi

In [9]:
import pandas as pd
print(f"Registros de auditoría: {len(AUDITORIA)}")
pd.DataFrame(AUDITORIA).tail(3)

Registros de auditoría: 4


,ts,usuario,hash_entrada,prediccion,probabilidad,modelo_version
1,2026-10-14T23:37:17.125276+00:00,cecilia,5703b08a9772e72ee1448bbec108433759625fb87a9467...,False,0.15,1
2,2026-10-14T23:37:17.165302+00:00,cecilia,5703b08a9772e72ee1448bbec108433759625fb87a9467...,False,0.15,1
3,2026-10-14T23:37:17.229866+00:00,cecilia,5703b08a9772e72ee1448bbec108433759625fb87a9467...,False,0.15,1


## 4. Reflexión
1. **¿Qué amenaza frena el rate limiting y por qué se relaciona con el robo del modelo?** Frena el abuso y el *scraping*: quien puede consultar sin límite reúne miles de pares entrada/salida y entrena una copia del modelo (**extracción/robo del modelo**), además de poder saturar el servicio (DoS). Limitar consultas por usuario encarece ese ataque.
2. **¿Qué elemento de SAIF cubre mi control principal?** JWT + roles + rate limiting → *bases de seguridad sólidas* (autenticación/autorización) y *automatizar defensas*. La auditoría y la model card → *contextualizar riesgos* y gobernanza/monitoreo.
3. **¿Por qué validar la entrada es también seguridad y no solo calidad?** Rechazar entradas malformadas o fuera de rango (tipos, `Literal`, `ge/le`) corta inyección, desbordes y entradas adversarias absurdas antes de llegar al modelo, y reduce la superficie de ataque.

### Límites conocidos
- El rate limiting es en memoria (un solo proceso); en producción iría en un API gateway o Redis.
- No hay TLS ni emisión real de tokens (`crear_token` es solo de demo): en producción lo emite un proveedor de identidad.
- La auditoría vive en memoria dentro del notebook; en producción iría a almacenamiento persistente e inmutable.